<a href="https://colab.research.google.com/github/HadzhymuradovaAnzhela/ab-testing-statistical-analysis/blob/main/Statistical_Analysis_of__AB_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Import

In [2]:
!pip install --upgrade google-cloud-bigquery
from google.colab import auth
from google.cloud import bigquery

import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

auth.authenticate_user()
client = bigquery.Client(project="data-analytics-mate")

query = """
WITH
  session_info AS(
  SELECT
    s.date,
    s.ga_session_id,
    sp.country,
    sp.device,
    sp.continent,
    sp.channel,
    ab.test,
    ab.test_group
  FROM
    `DA.ab_test` ab
  JOIN
    `DA.session` s
  ON
    ab.ga_session_id=s.ga_session_id
  JOIN
    `DA.session_params` sp
  ON
    ab.ga_session_id=sp.ga_session_id),
  session_with_orders AS(
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count (DISTINCT o.ga_session_id) AS session_with_order
  FROM
    `DA.order`o
  JOIN
    session_info
  ON
    o.ga_session_id=session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group),
  events AS(
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count (ep.ga_session_id) AS event_cnt,
    ep.event_name
  FROM
    `DA.event_params` ep
  JOIN
    session_info
  ON
    ep.ga_session_id=session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    ep.event_name),
  session AS(
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
  FROM
    session_info
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group),
  account AS(
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
  FROM
    `DA.account_session` acs
  JOIN
    session_info
  ON
    acs.ga_session_id=session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group)
SELECT
  session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session with orders' AS event_name,
  session_with_orders.session_with_order AS value
FROM
  session_with_orders
UNION ALL
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  events.event_cnt AS value
FROM
  events
UNION ALL
SELECT
  session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' AS event_name,
  session.session_cnt AS value
FROM
  session
UNION ALL
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' AS event_name,
  account.new_account_cnt AS value
FROM
  account
"""
query_job = client.query(query)
results = query_job.result()
df = results.to_dataframe()
df['event_name'] = df['event_name'].str.replace(' ', '_')
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new_account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new_account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new_account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new_account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new_account,1


**Code Description**
1. Data Grouping and Transformation  
The initial dataset was cleaned and grouped by required parameters. To prepare the data for analysis, it was split into two separate tables:
- A metrics table containing events: add_payment_info, add_shipping_info, begin_checkout, and new_accounts.
-  A sessions table containing traffic data.
This separation was necessary to align the data by index, as all metrics were originally stored in a single column.
2. Merging and Segmentation  
After aligning the indexes, the two tables were merged into a single dataset. This dataset was then divided into two subsets based on the test groups to isolate the performance of each variant.
3. Group Comparison and Metric Calculation  
The data was then re-combined to display metrics for Group 1 and Group 2 side-by-side. During this step, additional columns were added to the table, including: сlear metric names, conversion rate calculations for each group,  the percentage difference between the groups.
4. Statistical Testing  
A Python loop was used to iterate through every row of the final table. For each row, a Z-Test for Proportions was performed using the proportions_ztest function from the statsmodels library. The script calculated the p-value and determined whether the result was statistically significant based on the alpha level (0.05). This step added a clear conclusion for each specific metric and segment.
5. Final Formatting  
The last stage involved organizing the column order and renaming them to ensure the final output is clean, readable.

In [6]:
df_2 = df.groupby(['test', 'test_group', 'event_name'])['value'].sum().reset_index()
numerator = df_2[df_2['event_name'].isin(['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_account'])]
denominator =df_2[df_2['event_name'] == 'session'][['test', 'test_group', 'value']]
denominator.columns = ['test','test_group', 'session_count']
metrics_df = numerator.merge(denominator, on=['test', 'test_group'])
test_gr1 = metrics_df[metrics_df['test_group'] == 1].copy()
test_gr2 = metrics_df[metrics_df['test_group'] == 2].copy()
result = test_gr1.merge(test_gr2, on=['test', 'event_name'], suffixes=('_g1', '_g2'))
result['denominator'] = 'session'
result['metric'] = result['event_name'] + ' / ' + result['denominator']
result['conversion_rate_1'] = result['value_g1'] / result['session_count_g1']
result['conversion_rate_2'] = result['value_g2'] / result['session_count_g2']
result['metric_change'] = ((result['conversion_rate_2'] / result['conversion_rate_1'])-1)*100
stats = []
for i, row in result.iterrows():
    counts = [row['value_g2'], row['value_g1']]
    nobs = [row['session_count_g2'], row['session_count_g1']]
    z_stat, p_val = proportions_ztest(counts, nobs)
    stats.append([z_stat, p_val])
result[['z_stat', 'p_value']] = stats
result['significant'] = result['p_value'] < 0.05
final = result[[
    'test',
    'metric',
    'event_name',
    'denominator',
    'value_g1',
    'session_count_g1',
    'conversion_rate_1',
    'value_g2',
    'session_count_g2',
    'conversion_rate_2',
    'metric_change',
    'z_stat',
    'p_value',
    'significant']]
final = final.rename(columns={
    'event_name': 'numerator_ev',
    'denumer': 'denominator_ev',
    'value_g1': 'numerator_gr1',
    'session_count_g1': 'denominator_gr1',
    'value_g2': 'numerator_gr2',
    'session_count_g2': 'denominator_gr2'})
final.head()

,test,metric,numerator_ev,denominator,numerator_gr1,denominator_gr1,conversion_rate_1,numerator_gr2,denominator_gr2,conversion_rate_2,metric_change,z_stat,p_value,significant
0,1,add_payment_info / session,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info / session,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout / session,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
3,1,new_account / session,new_account,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info / session,add_payment_info,session,2344,50637,0.04629,2409,50244,0.047946,3.576911,1.240994,0.214608,False


**Geographic Analysis by Continent**  
The same analytical approach was also applied to examine the results across different continents.  
Segmenting the data by geography allows for the identification of specific regions where the test achieved success. Since user behavior often varies by market, a result that shows no significance globally might be highly effective in one particular continent. This granular view enables the business to implement tailored strategies and deploy innovations selectively in the regions where they are most successful, optimizing resources and impact.

In [7]:
df_3 = df.groupby(['test', 'continent', 'test_group', 'event_name'])['value'].sum().reset_index()
numerator_cont = df_3[df_3['event_name'].isin(['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_account'])]
denominator_cont =df_3[df_3['event_name'] == 'session'][['test','continent', 'test_group', 'value']]
denominator_cont.columns = ['test', 'continent', 'test_group', 'session_count']
metrics_df_cont = numerator_cont.merge(denominator_cont, on=['test','continent', 'test_group'])
test_gr1_c = metrics_df_cont[metrics_df_cont['test_group'] == 1].copy()
test_gr2_c = metrics_df_cont[metrics_df_cont['test_group'] == 2].copy()
result_cont = test_gr1_c.merge(test_gr2_c, on=['test','continent', 'event_name'], suffixes=('_g1', '_g2'))
result_cont['denominator'] = 'session'
result_cont['metric'] = result_cont['event_name'] + ' / ' + result_cont['denominator']
result_cont['conversion_rate_1'] = result_cont['value_g1'] / result_cont['session_count_g1']
result_cont['conversion_rate_2'] = result_cont['value_g2'] / result_cont['session_count_g2']
result_cont['metric_change'] = ((result_cont['conversion_rate_2'] / result_cont['conversion_rate_1'])-1)*100
stats = []
for i, row in result_cont.iterrows():
    counts = [row['value_g2'], row['value_g1']]
    nobs = [row['session_count_g2'], row['session_count_g1']]
    z_stat, p_val = proportions_ztest(counts, nobs)
    stats.append([z_stat, p_val])
result_cont[['z_stat', 'p_value']] = stats
result_cont['significant'] = result_cont['p_value'] < 0.05
final_cont = result_cont[[
    'test',
    'continent',
    'metric',
    'event_name',
    'denominator',
    'value_g1',
    'session_count_g1',
    'conversion_rate_1',
    'value_g2',
    'session_count_g2',
    'conversion_rate_2',
    'metric_change',
    'z_stat',
    'p_value',
    'significant']]
final_cont = final_cont.rename(columns={
    'event_name': 'numerator_ev',
    'denumer': 'denominator_ev',
    'value_g1': 'numerator_gr1',
    'session_count_g1': 'denominator_gr1',
    'value_g2': 'numerator_gr2',
    'session_count_g2': 'denominator_gr2'})
final_cont.head()


,test,continent,metric,numerator_ev,denominator,numerator_gr1,denominator_gr1,conversion_rate_1,numerator_gr2,denominator_gr2,conversion_rate_2,metric_change,z_stat,p_value,significant
0,1,(not set),add_payment_info / session,add_payment_info,session,7,97,0.072165,10,100,0.1,38.571429,0.695585,0.486689,False
1,1,(not set),add_shipping_info / session,add_shipping_info,session,4,97,0.041237,13,100,0.13,215.25,2.218144,0.026545,True
2,1,(not set),begin_checkout / session,begin_checkout,session,4,97,0.041237,22,100,0.22,433.5,3.706053,0.000211,True
3,1,(not set),new_account / session,new_account,session,7,97,0.072165,11,100,0.11,52.428571,0.921405,0.356839,False
4,1,Africa,add_payment_info / session,add_payment_info,session,19,494,0.038462,24,474,0.050633,31.64557,0.918808,0.358196,False


**Analysis by Device Type**  

The same analytical pipeline was applied to evaluate results across different device segments: Desktop, Mobile, and Tablet.  
Analyzing data at the device level is crucial because high-level results can be misleading. While the overall dataset might show no statistical significance, specific segments—such as Desktop users—may demonstrate a significant lift. Identifying these "winning" segments allows for more targeted strategies, enabling the implementation of innovations or new features specifically for the platforms where they provide the most value.

In [5]:
df_4 = df.groupby(['test', 'device', 'test_group', 'event_name'])['value'].sum().reset_index()
numerator_dev = df_4[df_4['event_name'].isin(['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_account'])]
denominator_dev =df_4[df_4['event_name'] == 'session'][['test','device', 'test_group', 'value']]
denominator_dev.columns = ['test', 'device', 'test_group', 'session_count']
metrics_df_dev = numerator_dev.merge(denominator_dev, on=['test','device', 'test_group'])
test_gr1_d = metrics_df_dev[metrics_df_dev['test_group'] == 1].copy()
test_gr2_d = metrics_df_dev[metrics_df_dev['test_group'] == 2].copy()
result_dev = test_gr1_d.merge(test_gr2_d, on=['test','device', 'event_name'], suffixes=('_g1', '_g2'))
result_dev['denominator'] = 'session'
result_dev['metric'] = result_dev['event_name'] + ' / ' + result_dev['denominator']
result_dev['conversion_rate_1'] = result_dev['value_g1'] / result_dev['session_count_g1']
result_dev['conversion_rate_2'] = result_dev['value_g2'] / result_dev['session_count_g2']
result_dev['metric_change'] = ((result_dev['conversion_rate_2'] / result_dev['conversion_rate_1'])-1)*100
stats = []
for i, row in result_dev.iterrows():
    counts = [row['value_g2'], row['value_g1']]
    nobs = [row['session_count_g2'], row['session_count_g1']]
    z_stat, p_val = proportions_ztest(counts, nobs)
    stats.append([z_stat, p_val])
result_dev[['z_stat', 'p_value']] = stats
result_dev['significant'] = result_dev['p_value'] < 0.05
final_dev = result_dev[[
    'test',
    'device',
    'metric',
    'event_name',
    'denominator',
    'value_g1',
    'session_count_g1',
    'conversion_rate_1',
    'value_g2',
    'session_count_g2',
    'conversion_rate_2',
    'metric_change',
    'z_stat',
    'p_value',
    'significant']]
final_dev = final_dev.rename(columns={
    'event_name': 'numerator_ev',
    'denumer': 'denominator_ev',
    'value_g1': 'numerator_gr1',
    'session_count_g1': 'denominator_gr1',
    'value_g2': 'numerator_gr2',
    'session_count_g2': 'denominator_gr2'})

final_dev.to_csv('ab_test_dev.csv', index=False)
final_dev.head()

,test,device,metric,numerator_ev,denominator,numerator_gr1,denominator_gr1,conversion_rate_1,numerator_gr2,denominator_gr2,conversion_rate_2,metric_change,z_stat,p_value,significant
0,1,desktop,add_payment_info / session,add_payment_info,session,1130,26467,0.042695,1256,26417,0.047545,11.360819,2.686998,0.007210,True
1,1,desktop,add_shipping_info / session,add_shipping_info,session,1711,26467,0.064647,1916,26417,0.072529,12.193247,3.586024,0.000336,True
2,1,desktop,begin_checkout / session,begin_checkout,session,2108,26467,0.079646,2404,26417,0.091002,14.257595,4.673980,0.000003,True
3,1,desktop,new_account / session,new_account,session,2203,26467,0.083236,2147,26417,0.081273,-2.357527,-0.821212,0.411526,False
4,1,mobile,add_payment_info / session,add_payment_info,session,810,17896,0.045262,942,17767,0.05302,17.140683,3.389330,0.000701,True


[Link to Tableau dashboard](https://public.tableau.com/app/profile/angela.krupa/viz/ABTestDistributionandAnalysis/ABtest#2)  
**Data Sources for Visualization**  
[General Statistics Data](https://drive.google.com/file/d/1EK_CJmPMVCdt6qFFs_i7gZwj_yxTC0U6/view?usp=sharing)  
[Device-Level Analysis Data](https://drive.google.com/file/d/16LrcbEsjo69hEeKKBL6aNwDdZQEM3rTG/view?usp=sharing)  
[Regional Performance](https://drive.google.com/file/d/1grdbXgCByn8p9i_hHzyuS_tIhYKh5vaX/view?usp=sharing)

# Conclusions

## **Dashboard description**  
The first page of the dashboard provides a comprehensive overview of traffic distribution across test groups, demonstrating that traffic is evenly balanced across countries, continents, devices, marketing channels, and time series while also highlighting percentage changes in key metrics.   
Complementing this, the second page presents the results of statistical significance testing at the total level as well as broken down by continents and devices, featuring interactive filters that allow users to toggle between specific tests, devices, or geographic regions for a more granular evaluation.  
## **Test 1**  
**Overall Performance**  
The traffic distribution for Test 1 is balanced across all segments. General metric analysis shows a significant increase in add_payment_info, add_shipping_info, and begin_checkout, alongside a decrease in new_accounts. Statistical significance testing confirms that the growth in payment, shipping, and checkout metrics for Group 2 is statistically significant, whereas the decline in new account registrations is not. This indicates a verified positive impact on the core conversion funnel.  
**Device-Level Insights**  
The positive trend is most prominent among Desktop users, whose performance mirrors the overall significant growth. Conversely, results for Mobile and Tablet users are less favorable. Most mobile metrics lack statistical significance, with only add_payment_info showing a confirmed positive increase. Tablet users demonstrated an overall decline in performance. Based on these findings, it is recommended to implement the changes for the Desktop version immediately. For Mobile users, the sample size should be increased to reach a definitive conclusion, while for Tablets, an alternative optimization strategy is required.  
**Geographic Segmentation**  
The strongest performance was observed in Europe, where the results are fully consistent with the overall positive statistical significance. In Asia and America, the majority of metrics remain statistically insignificant, suggesting that further data collection is necessary. Notably, in America, the decrease in new account registrations was found to be statistically significant. Due to insufficient sample sizes, definitive conclusions cannot yet be drawn for Oceania and Africa.  
**Final Recommendation**  
The test was most successful among European and Desktop users. Therefore, the most optimal and data-driven decision is to roll out the changes specifically for these high-performing segments while continuing observation or refining strategies for other groups.  
##**Test 2**  
**Overall Performance**  
Traffic distribution remains balanced between groups. While there is a noticeable increase in add_to_cart and add_payment_info rates, other engagement metrics—such as item views and overall user activity—have declined. Despite the positive trend in funnel progression, the changes in conversion for payment info, shipping info, checkout initiation, and account creation are not statistically significant at the total level.  
**Device-Level Insights**  
Segmenting by device reveals that the only statistically significant improvement is the increase in begin_checkout conversion among Desktop users. For all other devices, the observed changes lack mathematical significance, suggesting the test had a minimal impact across most platforms.  
**Geographic Segmentation**  
Results vary significantly by region. Africa shows statistically significant negative results, indicating the current strategy is unsuccessful there. Oceania reports positive, partially significant trends, but the small sample size requires further data collection. In North America, only add_payment_info saw a significant boost, while Europe experienced a significant decrease in that same metric. Asia shows generally positive trends, but only the increase in begin_checkout is statistically confirmed.  
**Final Recommendation**   
Implementing the changes is not recommended at this stage. The lack of consistent statistical significance across key metrics and the negative impact in certain regions suggest the strategy needs refinement. Further investigation into user behavior and a revised approach are necessary before additional testing.  
##**Test 3**  
**Overall Performance**  
Traffic is evenly distributed across all segments. General indicators show a slight increase in add_payment_info and the number of sessions with orders. However, there is a noticeable decline in add_shipping_info, add_to_cart, and begin_checkout. Regarding statistical significance, the only confirmed result at the total level is a decrease in the conversion rate for beginning checkout. While payment info conversion showed a slight uptick, it did not reach the threshold for significance, and the declines in shipping info and account creation remain statistically unconfirmed.  
**Device-Level Insights**  
The device-level data mirrors the overall negative trend. Statistical significance was confirmed only for the decrease in begin_checkout conversion among Mobile and Tablet users. All other metrics across all device types are statistically insignificant, with the majority of indicators reflecting a negative impact from the tested changes.  
**Geographic Segmentation**  
The regional analysis confirms the lack of positive impact. In North America, there is a statistically significant decrease in the conversion rate for beginning checkout. Across all other continents, metrics remain statistically unconfirmed, and the general trend remains largely negative, showing no evidence of success in any specific market.  
**Final Recommendation**  
Based on the confirmed negative impact on the checkout funnel and the lack of significant gains in other areas, these changes should not be implemented. The test results suggest that the current modifications are hindering the user journey, particularly at the checkout stage. It is recommended to revert to the original version and re-evaluate the hypothesis.  
##**Test 4**   
**Overall Performance**  
Traffic distribution is balanced across all segments. However, the data indicates that the current strategy is largely ineffective, as there is a general decline in add_payment_info, add_shipping_info, begin_checkout, and new_accounts. Statistical significance testing confirms a verified decrease in both checkout initiation and new account creation. While the declines in payment and shipping info conversion are visible, they have not yet reached the threshold of statistical significance at the total level.  
**Device-Level Insights**  
The impact varies significantly by device type. Desktop users show statistically significant negative results across all four key metrics, and Tablets mirror this downward trend. Interestingly, Mobile users represent a unique segment where add_payment_info, add_shipping_info, and begin_checkout conversions actually show positive movement. Specifically, the increase in begin_checkout for mobile is statistically significant. The number of new accounts for mobile users decreased, though not significantly.  
**Geographic Segmentation**  
Regional analysis confirms a statistically significant performance drop in Europe. Results for North America remain neutral and lack statistical confirmation. In Asia, there is a confirmed decrease in new_accounts conversion, while other metrics remain inconclusive. Oceania shows a verified decline in performance, though the small sample size suggests caution. Africa shows some positive signs, including a significant increase in checkout initiation, but like Oceania, the sample size is currently too small for a definitive conclusion.  
**Final Recommendation**  
The current rollout should be halted due to its negative impact on the majority of segments. However, the positive signals from the Mobile segment suggest a potential opportunity. After refining the strategy, a separate, targeted test for mobile users with a larger sample size should be conducted to confirm if these gains can be sustained. For all other segments, an entirely different strategic approach is required.  

